# Ch2. Time Series Graphics
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [ ]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast

## [Slide 2] 2.1 The Nixtlaverse Data Format

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "unique_id": "beer",
    "ds": pd.period_range("2000Q1", periods=10, freq="Q").to_timestamp(),
    "y": [443, 410, 420, 532, 433, 421, 410, 512, 449, 381],
})
print(df.head(3))

## [Slide 4] 2.1 Timestamps and Periods

In [ ]:
import pandas as pd

# Timestamp: an instant in time
pd.Timestamp("2020-01")          # Timestamp('2020-01-01 00:00:00')

# Period: a span of time
pd.Period("2020-01", freq="M")   # Period('2020-01', 'M')

# Create sequences
ts_range = pd.date_range("2020-01-01", "2020-12-01", freq="MS")
p_range  = pd.period_range("2020Q1", periods=8, freq="Q")

# Convert Period -> Timestamp for plotting
p_range.to_timestamp()

## [Slide 6] 2.1 Working with Time Series DataFrames

In [ ]:
import pandas as pd

# Filter to one series
a10 = pbs.loc[pbs["ATC2"] == "A10"].copy()

# Remove unused columns
a10 = a10.drop(columns=["ATC1", "ATC2"])

# Aggregate: total cost across all products
total_cost = (
    pbs
    .groupby("Month", as_index=False)
    .agg({"Cost": "sum"})
    .assign(Cost=lambda x: (x["Cost"] / 1e6).round(2))
)

# Read CSV with automatic date parsing
prison = pd.read_csv(
    "data/prison_population.csv",
    parse_dates=["Date"]
)

## [Slide 8] 2.2 Time Plots

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd, numpy as np

# Generate synthetic monthly series
dates = pd.date_range("2010-01", periods=120, freq="MS")
y = (100 + np.arange(120)*0.3
     + 8*np.sin(2*np.pi*np.arange(120)/12)
     + np.random.default_rng(1).normal(0, 2, 120))

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(dates, y, lw=1.2, color="steelblue")
ax.set(xlabel="Month", ylabel="Value",
       title="Monthly time series (2010-2019)")
plt.tight_layout(); plt.show()

## [Slide 10] 2.2 Using plot_series (utilsforecast)

In [ ]:
import pandas as pd
from utilsforecast.plotting import plot_series

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
ansett = pd.read_csv(BASE + "ansett.csv", parse_dates=["ds"])

# 컬럼 구조 확인
print(ansett.columns.tolist())      # 컬럼 목록
print(ansett["Airports"].unique())  # 노선 목록
print(ansett.head(3))               # 데이터 미리보기

# Economy 클래스만 필터링 → unique_id / ds / y 형식으로 변환
df = ansett[ansett["Class"] == "Economy"].copy()
df = df.rename(columns={"Airports": "unique_id"})[["unique_id", "ds", "y"]]

# 다중 시계열 시각화 (노선별 서브플롯)
plot_series(df, id_col="unique_id", time_col="ds", target_col="y")

## [Slide] 2.2 다중 시계열 겹쳐 그리기

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
ansett = pd.read_csv(BASE + "ansett.csv", parse_dates=["ds"])

# Economy 클래스만 필터링
df = ansett[ansett["Class"] == "Economy"].copy()
df = df.rename(columns={"Airports": "unique_id"})[["unique_id", "ds", "y"]]

# 여러 시계열을 같은 축에 겹쳐 그리기
fig, ax = plt.subplots()
for uid, grp in df.groupby("unique_id"):
    ax.plot(grp["ds"], grp["y"], label=uid, lw=1)
ax.set_xlabel("Week")
ax.set_ylabel("Passengers")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## [Slide 13] 2.4 Seasonal Plots

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

dates = pd.date_range("2000-01", periods=120, freq="MS")
y = (50 + 5*np.sin(2*np.pi*np.arange(120)/12)
     + np.random.default_rng(7).normal(0, 1, 120))
df = pd.DataFrame({"ds": dates, "y": y})
df["year"]  = df["ds"].dt.year
df["month"] = df["ds"].dt.month

fig, ax = plt.subplots(figsize=(7, 3.2))
for yr, g in df.groupby("year"):
    ax.plot(g["month"], g["y"], lw=1, alpha=0.7, label=str(yr))
ax.set(xlabel="Month", ylabel="Value",
       title="Seasonal plot: monthly series")
ax.set_xticks(range(1, 13))
plt.tight_layout(); plt.show()

## [Slide 16] 2.5 Seasonal Subseries Plots

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt

dates = pd.date_range("2005-01", periods=180, freq="MS")
y = (50 + 5*np.sin(2*np.pi*np.arange(180)/12)
     + 0.15*np.arange(180)
     + np.random.default_rng(3).normal(0, 1.5, 180))
df = pd.DataFrame({"ds": dates, "y": y,
                   "month": dates.month, "year": dates.year})

fig, axes = plt.subplots(1, 12, figsize=(14, 3.2), sharey=True)
for m, ax in enumerate(axes, 1):
    sub = df[df["month"] == m]
    ax.plot(sub["year"], sub["y"], "o-", ms=3, lw=1)
    ax.axhline(sub["y"].mean(), color="blue", lw=1.5)
    ax.set_title(f"M{m}", fontsize=7)
    ax.tick_params(axis="x", labelsize=5, rotation=45)
plt.suptitle("Seasonal subseries plot", y=1.02)
plt.tight_layout(); plt.show()

## [Slide] 2.6 Scatterplots and Correlation

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
ins = pd.read_csv(BASE + "insurance.csv", parse_dates=["ds"])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# 시계열 플롯
axes[0].plot(ins["ds"], ins["Quotes"],    label="Quotes",    lw=1.5)
axes[0].plot(ins["ds"], ins["TVadverts"], label="TVadverts", lw=1.5, linestyle="--")
axes[0].set_title("Time Series"); axes[0].legend(fontsize=8)

# 산점도
axes[1].scatter(ins["TVadverts"], ins["Quotes"], alpha=0.7, s=30, color="steelblue")
axes[1].set(xlabel="TV Adverts", ylabel="Quotes", title="Quotes vs. TVadverts")

r = ins["Quotes"].corr(ins["TVadverts"])
axes[1].annotate(f"r = {r:.2f}", xy=(0.05, 0.90), xycoords="axes fraction", fontsize=10)

plt.tight_layout(); plt.show()

## [Slide 22] 2.6 Scatterplot Matrices

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
tour = pd.read_csv(BASE + "tourism.csv", parse_dates=["ds"])

# Long → Wide 변환: 주(State)별 Holiday 방문객 합산
tourism_wide = (
    tour[tour["Purpose"] == "Holiday"]
    .groupby(["ds", "State"])["y"].sum()
    .unstack("State")
)

sns.pairplot(
    tourism_wide,
    diag_kind="kde",
    plot_kws={"alpha": 0.3, "s": 10},
)
plt.suptitle("Quarterly Holiday Trips by State", y=1.01)
plt.tight_layout(); plt.show()

## [Slide 24] 2.7 Lag Plots

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/bcseong2/fpppy-labs/main/data/"
aus = pd.read_csv(BASE + "aus_production.csv", parse_dates=["ds"])
beer = aus[["ds", "Beer"]].rename(columns={"Beer": "y"})
beer = beer.sort_values("ds").reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, k in zip(axes.flat, range(1, 9)):
    ax.scatter(beer["y"].shift(k), beer["y"],
               alpha=0.5, s=10, c=beer.index % 4,
               cmap="tab10")
    ax.set(title=f"Lag {k}", xlabel=f"$y_{{t-{k}}}$", ylabel="$y_t$")
plt.tight_layout(); plt.show()

## [Slide 27] 2.8 Autocorrelation

In [ ]:
import statsmodels.api as sm

acf_vals = sm.tsa.acf(beer["y"], nlags=9, fft=False)
print(acf_vals)

## [Slide 29] 2.8 Plotting the ACF (Correlogram)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
plot_acf(
    beer["y"],
    lags=16,
    ax=ax,
    zero=False,          # omit lag-0 (always = 1)
    bartlett_confint=False,
)
ax.set(xlabel="Lag", ylabel="ACF",
       title="ACF: Quarterly Australian Beer Production")
plt.tight_layout(); plt.show()

## [Slide 33] 2.9 Generating and Testing White Noise

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

rng = np.random.default_rng(99)
wn = rng.standard_normal(50)   # white noise series

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

# Time plot
axes[0].plot(wn, lw=1, color="steelblue")
axes[0].axhline(0, color="gray", lw=0.8, ls="--")
axes[0].set(title="White noise series (T=50)",
            xlabel="t", ylabel="$e_t$")

# ACF
plot_acf(wn, lags=20, ax=axes[1], zero=False)
axes[1].set(title=f"ACF (bounds ≈ ±{1.96/50**0.5:.2f})")

plt.tight_layout(); plt.show()

## [Slide 35] 2.9 Residual Checking with ACF

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(wn, lags=[10], return_df=True)
print(lb)   # lb_stat, lb_pvalue